# Roof Measurement Extraction — Exploration Notebook

End-to-end walkthrough: generate synthetic LiDAR → segment planes → extract pitch / height / facet count → visualize results.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from roof_measurements.segmentation import segment_planes
from roof_measurements.features import compute_facet, compute_height, estimate_point_density
from roof_measurements.models import BuildingResult

print("Imports OK")

## 1. Synthetic Point Cloud Generators

In [ ]:
def make_gable(pitch_deg=30.0, width=10.0, length=10.0, n=1600, noise=0.02, seed=0):
    """Two-face gable roof. Ridge at Z = (width/2)*tan(pitch), centred at X=0."""
    rng = np.random.default_rng(seed)
    slope = np.tan(np.radians(pitch_deg))
    ridge_z = (width / 2) * slope
    pts = []
    for _ in range(n // 2):
        x = rng.uniform(-width / 2, 0)
        y = rng.uniform(0, length)
        z = ridge_z + x * slope          # x < 0 → z < ridge_z
        pts.append([x, y, z])
    for _ in range(n // 2):
        x = rng.uniform(0, width / 2)
        y = rng.uniform(0, length)
        z = ridge_z - x * slope
        pts.append([x, y, z])
    pts = np.array(pts)
    pts += rng.normal(0, noise, pts.shape)
    return pts, ridge_z



def make_hip(pitch_deg=25.0, lx=10.0, ly=8.0, n=2000, noise=0.02, seed=1):
    """Four-face hip roof with a horizontal ridge of length (lx - ly).

    Generates n/4 points per face so each planar facet is well-sampled.
    All four faces share the same pitch angle (standard hip roof geometry).

    Faces:
      front / back  -- trapezoidal main slopes (normal in +/-Y direction)
      left  / right -- triangular hip ends     (normal in +/-X direction)
    """
    assert lx >= ly, "lx must be >= ly (lx is the longer dimension)"
    rng = np.random.default_rng(seed)
    slope = np.tan(np.radians(pitch_deg))
    ridge_z    = (ly / 2) * slope          # eave-to-ridge height
    ridge_half = (lx - ly) / 2            # half-length of the flat ridge

    pts = []
    n4  = n // 4

    # Front main slope (y > 0, trapezoidal)
    # At fractional depth t in [0,1]: y = t*(ly/2), z = ridge_z*(1-t)
    # x-extent widens from ridge_half at t=0 to lx/2 at t=1
    for _ in range(n4):
        t = rng.uniform(0, 1)
        y = t * (ly / 2)
        x_half = ridge_half + t * (lx / 2 - ridge_half)
        x = rng.uniform(-x_half, x_half)
        pts.append([x,  y, ridge_z * (1 - t)])

    # Back main slope (y < 0, symmetric)
    for _ in range(n4):
        t = rng.uniform(0, 1)
        y = -t * (ly / 2)
        x_half = ridge_half + t * (lx / 2 - ridge_half)
        x = rng.uniform(-x_half, x_half)
        pts.append([x,  y, ridge_z * (1 - t)])

    # Right hip end (x > 0, triangular)
    # slope = ridge_z/(ly/2) == same pitch as main faces
    for _ in range(n4):
        t = rng.uniform(0, 1)
        x     = ridge_half + t * (ly / 2)
        y_half = (1 - t) * (ly / 2)
        y = rng.uniform(-y_half, y_half)
        pts.append([x,  y, ridge_z * (1 - t)])

    # Left hip end (x < 0, triangular)
    for _ in range(n4):
        t = rng.uniform(0, 1)
        x     = -(ridge_half + t * (ly / 2))
        y_half = (1 - t) * (ly / 2)
        y = rng.uniform(-y_half, y_half)
        pts.append([x,  y, ridge_z * (1 - t)])

    pts = np.array(pts)
    pts += rng.normal(0, noise, pts.shape)
    return pts, ridge_z

ef make_flat(width=8.0, length=6.0, drainage_deg=2.0, n=500, noise=0.01, seed=2):
    """Near-flat roof with slight drainage slope, Z ~ 3m."""
    rng = np.random.default_rng(seed)
    pts = []
    base_z = 3.0
    for _ in range(n):
        x = rng.uniform(0, width)
        y = rng.uniform(0, length)
        z = base_z + x * np.tan(np.radians(drainage_deg))
        pts.append([x, y, z])
    pts = np.array(pts)
    pts += rng.normal(0, noise, pts.shape)
    return pts, base_z


gable_pts, gable_ridge = make_gable()
hip_pts,   hip_ridge   = make_hip()
flat_pts,  flat_ridge  = make_flat()

print(f"Gable:  {len(gable_pts)} pts  |  expected ridge ~{gable_ridge:.2f} m")
print(f"Hip:    {len(hip_pts)} pts  |  expected ridge ~{hip_ridge:.2f} m")
print(f"Flat:   {len(flat_pts)} pts  |  expected ~flat (< 5°)")

## 2. Visualise Raw Point Clouds

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), subplot_kw={"projection": "3d"})

for ax, pts, title in zip(axes, [gable_pts, hip_pts, flat_pts], ["Gable", "Hip", "Flat"]):
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=1, c=pts[:, 2], cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_zlabel("Z (m)")
    ax.view_init(elev=25, azim=-60)

plt.tight_layout()
plt.show()

## 3. Segment Planes & Colour by Facet

In [ ]:
PALETTE = ["#e63946", "#2a9d8f", "#e9c46a", "#f4a261", "#264653", "#a8dadc"]

def plot_segmented(ax, facet_list, title, method):
    for i, pts in enumerate(facet_list):
        color = PALETTE[i % len(PALETTE)]
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=2, c=color,
                   label=f"Facet {i+1}")
    ax.set_title(f"{title}\n({method}, {len(facet_list)} facets)")
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.legend(markerscale=4, fontsize=7, loc="upper left")
    ax.view_init(elev=25, azim=-60)


configs = [
    (gable_pts, "Gable", 0.15),
    (hip_pts,   "Hip",   0.20),
    (flat_pts,  "Flat",  0.10),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), subplot_kw={"projection": "3d"})

results = {}
for ax, (pts, label, thresh) in zip(axes, configs):
    facets, method = segment_planes(pts, distance_threshold=thresh, min_facet_area_m2=1.0)
    results[label] = (facets, method)
    plot_segmented(ax, facets, label, method)

plt.tight_layout()
plt.show()

for label, (facets, method) in results.items():
    print(f"{label:6s}: {len(facets)} facets  [{method}]")

## 4. Feature Extraction per Facet

In [ ]:
import textwrap

def summarise(label, pts, ground_z, thresh=0.15):
    facets, method = segment_planes(pts, distance_threshold=thresh, min_facet_area_m2=1.0)
    all_pts = np.vstack(facets)
    height_m, ridge_m = compute_height(all_pts, ground_z)

    print(f"\n{'='*55}")
    print(f"  {label}  |  method={method}  |  facets={len(facets)}")
    print(f"  Ground: {ground_z:.2f} m  |  Ridge: {ridge_m:.2f} m  |  Height: {height_m:.2f} m")
    print(f"{'='*55}")
    for i, fpts in enumerate(facets):
        f = compute_facet(i + 1, fpts)
        flat_tag = " [FLAT]" if f.is_flat else ""
        print(
            f"  Facet {f.facet_id}: pitch={f.pitch_deg:5.1f}°  "
            f"azimuth={f.azimuth_deg:5.1f}°  "
            f"area={f.area_m2:6.1f} m²  "
            f"pts={f.num_points:5d}{flat_tag}"
        )


summarise("Gable (expected 30°, 2 facets)",  gable_pts, ground_z=0.0, thresh=0.15)
summarise("Hip   (expected 25°, 4 facets)",  hip_pts,   ground_z=0.0, thresh=0.20)
summarise("Flat  (expected  2°, 1 facet )",  flat_pts,  ground_z=0.0, thresh=0.10)

## 5. Full Pipeline — Write a LAS file & run `process_file()`

In [ ]:
import laspy
from pathlib import Path
from roof_measurements.pipeline import process_file

TMP = Path("/tmp/roof_test")
TMP.mkdir(exist_ok=True)

def write_las(pts, path, classification=6):
    """Write a numpy (N,3) array as a LAS 1.4 file with given ASPRS class."""
    las = laspy.LasData(laspy.LasHeader(version="1.4", point_format=6))
    las.x = pts[:, 0]
    las.y = pts[:, 1]
    las.z = pts[:, 2]
    las.classification = np.full(len(pts), classification, dtype=np.uint8)
    las.write(str(path))
    return path

gable_las = write_las(gable_pts, TMP / "gable.las")
hip_las   = write_las(hip_pts,   TMP / "hip.las")
flat_las  = write_las(flat_pts,  TMP / "flat.las")

print("LAS files written to", TMP)

In [ ]:
import json

for las_path, label in [(gable_las, "gable"), (hip_las, "hip"), (flat_las, "flat")]:
    result = process_file(las_path, building_id=label)
    print(result.model_dump_json(indent=2))

## 6. Accuracy Summary — Measured vs Expected

In [ ]:
ground_zs = {"gable": 0.0, "hip": 0.0, "flat": 0.0}
expected = {
    "gable": {"facets": 2, "pitch": 30.0, "height": gable_ridge},
    "hip":   {"facets": 4, "pitch": 25.0, "height": hip_ridge},
    "flat":  {"facets": 1, "pitch":  2.0, "height": flat_ridge},
}

print(f"{'Roof':6s}  {'Exp.F':>6}  {'Got.F':>6}  {'Exp.P°':>7}  {'Got.P°':>7}  {'Δ P°':>6}  {'Exp.H':>6}  {'Got.H':>6}  {'Δ H':>6}")
print("-" * 70)

for las_path, label in [(gable_las, "gable"), (hip_las, "hip"), (flat_las, "flat")]:
    r = process_file(las_path, building_id=label)
    exp = expected[label]

    # Mean pitch across facets
    mean_pitch = np.mean([f.pitch_deg for f in r.facets])
    delta_pitch = mean_pitch - exp["pitch"]
    delta_height = r.height_m - exp["height"]

    ok_f = "✓" if r.num_facets == exp["facets"] else "✗"
    ok_p = "✓" if abs(delta_pitch) < 2.0 else "✗"
    ok_h = "✓" if abs(delta_height) < 0.5 else "✗"

    print(
        f"{label:6s}  {exp['facets']:>6}  {r.num_facets:>5}{ok_f}  "
        f"{exp['pitch']:>7.1f}  {mean_pitch:>6.1f}{ok_p}  {delta_pitch:>+6.2f}  "
        f"{exp['height']:>6.2f}  {r.height_m:>6.2f}{ok_h}  {delta_height:>+6.2f}"
    )

> **Note on height accuracy:**  
> The `flat` row shows a large height error (Δ≈−2.71m). This is expected: the synthetic LAS files contain **only roof points (class 6)** — no ground points (class 2). Without ground points, `io.py` falls back to `min(Z_building)` as the ground reference, which for a flat roof at Z≈3m gives height≈0.28m instead of 3m.  
> **Fix for real data:** ensure your LAS files include class 2 (ground) points, or pass a known ground elevation via the pipeline API. The gable result is accurate because the ridge is well above the eave, so `min(Z) ≈ eave` still gives a meaningful height.

## 7. Pitch Sensitivity — vary pitch angle, check recovery accuracy

In [ ]:
from roof_measurements.features import compute_facet

pitch_angles = [10, 15, 20, 25, 30, 35, 40, 45]
measured = []
errors = []

for p in pitch_angles:
    pts, _ = make_gable(pitch_deg=p, n=1600, noise=0.02)
    facets, _ = segment_planes(pts, distance_threshold=0.15, min_facet_area_m2=1.0)
    pitches = [compute_facet(i+1, f).pitch_deg for i, f in enumerate(facets)]
    mean_p = float(np.mean(pitches)) if pitches else float("nan")
    measured.append(mean_p)
    errors.append(mean_p - p)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(pitch_angles, pitch_angles, "k--", label="Perfect", alpha=0.5)
ax1.plot(pitch_angles, measured, "o-", color="#e63946", label="Measured")
ax1.set_xlabel("True pitch (°)")
ax1.set_ylabel("Measured pitch (°)")
ax1.set_title("Pitch Recovery")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.bar(pitch_angles, errors, color=["#2a9d8f" if abs(e) < 2 else "#e63946" for e in errors], width=3)
ax2.axhline(0, color="k", linewidth=0.8)
ax2.axhline(2, color="gray", linestyle="--", linewidth=0.8, label="±2° target")
ax2.axhline(-2, color="gray", linestyle="--", linewidth=0.8)
ax2.set_xlabel("True pitch (°)")
ax2.set_ylabel("Error (°)")
ax2.set_title("Pitch Error (green = within ±2°)")
ax2.legend()
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print(f"Max absolute error: {max(abs(e) for e in errors):.2f}°")

## 8. Load Your Own LAS/LAZ File

Replace the path below with a real file to test the full pipeline end-to-end.

In [ ]:
# ── Edit this path ──────────────────────────────────────────────
LAS_FILE = "/home/dhruv/repos/roof_measurements/data/nyc_sandy.laz"
# ────────────────────────────────────────────────────────────────

from pathlib import Path
if Path(LAS_FILE).exists():
    result = process_file(
        LAS_FILE,
        distance_threshold=0.15,   # RANSAC inlier tolerance (m)
        min_facet_area_m2=1.0,     # drop facets smaller than 1 m²
        max_planes=20,
    )
    print(result.model_dump_json(indent=2))

    # 3D plot of results
    fig = plt.figure(figsize=(9, 6))
    ax = fig.add_subplot(111, projection="3d")
    from roof_measurements.io import load_building_points
    pts, _ = load_building_points(LAS_FILE)
    facets, _ = segment_planes(pts, distance_threshold=0.15, min_facet_area_m2=1.0)
    for i, fpts in enumerate(facets):
        ax.scatter(fpts[:, 0], fpts[:, 1], fpts[:, 2], s=1,
                   c=PALETTE[i % len(PALETTE)], label=f"Facet {i+1}")
    ax.set_title(f"{Path(LAS_FILE).name}  —  {result.num_facets} facets, height={result.height_m:.1f} m")
    ax.legend(markerscale=5)
    plt.tight_layout()
    plt.show()
else:
    print(f"File not found: {LAS_FILE!r}  — update the path above to use a real file.")

## 9. Real LiDAR — Austin TX (USGS 3DEP QL2, 2017)

`austin_buildings_sample.laz` is a 150×150 m clip from a USGS 3DEP QL2 tile over Austin, TX.  
Source: [USGS TNM — TX Central B1 2017](https://www.usgs.gov/the-national-map-data-delivery)  
Classes present: 2 Ground · 3 Low Veg · 4 Med Veg · 5 High Veg · **6 Building** · 7 Noise  
Point density: ~6.7 pts/m²

In [ ]:
import laspy, numpy as np

LAS_SAMPLE = "../data/austin_buildings_sample.laz"

with laspy.open(LAS_SAMPLE) as f:
    las = f.read()

xyz = np.column_stack([las.x.scaled_array(), las.y.scaled_array(), las.z.scaled_array()])
cls = np.array(las.classification)

names = {1:"Unclassified", 2:"Ground", 3:"Low Veg", 4:"Med Veg", 5:"High Veg", 6:"Building", 7:"Noise"}
classes, counts = np.unique(cls, return_counts=True)

print(f"{'Class':>6}  {'Name':<15}  {'Points':>10}  {'%':>6}")
print("-" * 45)
for c, n in zip(classes, counts):
    print(f"  {c:>4}  {names.get(int(c),'Other'):<15}  {n:>10,}  {100*n/len(cls):>5.1f}%")
print(f"\nTotal: {len(xyz):,} pts  |  Area: 150×150 m  |  Density: {len(xyz)/150**2:.1f} pts/m²")
print(f"Z range: {xyz[:,2].min():.1f} – {xyz[:,2].max():.1f} m")

In [ ]:
# Visualise all classes in the raw scene
class_colors = {1:"#aaa", 2:"#8B6914", 3:"#90EE90", 4:"#228B22", 5:"#006400", 6:"#e63946", 7:"#ff00ff"}

fig = plt.figure(figsize=(14, 5))

# 3D full scene
ax1 = fig.add_subplot(121, projection="3d")
for c in np.unique(cls):
    m = cls == c
    col = class_colors.get(int(c), "#888")
    ax1.scatter(xyz[m,0]-xyz[:,0].min(), xyz[m,1]-xyz[:,1].min(), xyz[m,2],
                s=0.3, c=col, label=names.get(int(c), str(c)), alpha=0.7)
ax1.set_title("Austin TX — Full Scene\n(red = Building class 6)")
ax1.set_xlabel("X (m)"); ax1.set_ylabel("Y (m)"); ax1.set_zlabel("Z (m)")
ax1.legend(markerscale=10, fontsize=7, loc="upper left")
ax1.view_init(elev=30, azim=-60)

# Top-down building footprint
ax2 = fig.add_subplot(122)
bldg = xyz[cls == 6]
ground = xyz[cls == 2]
ax2.scatter(ground[:,0]-xyz[:,0].min(), ground[:,1]-xyz[:,1].min(), s=0.2, c="#8B6914", alpha=0.3, label="Ground")
ax2.scatter(bldg[:,0]-xyz[:,0].min(), bldg[:,1]-xyz[:,1].min(), s=0.5, c="#e63946", alpha=0.8, label="Building")
ax2.set_title("Top-down view — Building footprints")
ax2.set_xlabel("X (m)"); ax2.set_ylabel("Y (m)")
ax2.legend(markerscale=5)
ax2.set_aspect("equal")

plt.tight_layout()
plt.show()

In [ ]:
# Run the full pipeline on the real data
from roof_measurements.pipeline import process_file

result = process_file(LAS_SAMPLE, building_id="austin_tx_sample")
print(result.model_dump_json(indent=2))

In [ ]:
# Visualise segmented facets on the real data
from roof_measurements.segmentation import segment_planes
from roof_measurements.io import load_building_points

bldg_pts, ground_z = load_building_points(LAS_SAMPLE)
facets, method = segment_planes(bldg_pts, distance_threshold=0.2, min_facet_area_m2=1.0)

fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(121, projection="3d")
for i, pts in enumerate(facets):
    c = PALETTE[i % len(PALETTE)]
    ax1.scatter(pts[:,0]-bldg_pts[:,0].min(), pts[:,1]-bldg_pts[:,1].min(), pts[:,2],
                s=1, c=c, label=f"Facet {i+1}")
ax1.set_title(f"Real Data Segmentation\n({method}, {len(facets)} facets)")
ax1.set_xlabel("X"); ax1.set_ylabel("Y"); ax1.set_zlabel("Z (m)")
if len(facets) <= 10:
    ax1.legend(markerscale=5, fontsize=7)
ax1.view_init(elev=35, azim=-50)

# Top-down
ax2 = fig.add_subplot(122)
for i, pts in enumerate(facets):
    c = PALETTE[i % len(PALETTE)]
    ax2.scatter(pts[:,0]-bldg_pts[:,0].min(), pts[:,1]-bldg_pts[:,1].min(),
                s=1, c=c, alpha=0.7)
ax2.set_title("Top-down facet map")
ax2.set_xlabel("X (m)"); ax2.set_ylabel("Y (m)")
ax2.set_aspect("equal")

plt.tight_layout()
plt.show()

print(f"\nDetected {len(facets)} facets via {method}")
print(f"Ground elevation: {ground_z:.2f} m  |  Height: {result.height_m:.2f} m")

## 10. OSM Building Footprints — Clip LiDAR Per Building

Instead of running RANSAC on an entire city tile, fetch building footprints from OpenStreetMap, clip the point cloud to each footprint, then process each building independently.

**EPSG codes for the sample files:**
- `austin_buildings_sample.laz` → EPSG:32614 (UTM Zone 14N)
- `nyc_sandy.laz` → EPSG:32618 (UTM Zone 18N)

In [ ]:
from roof_measurements.footprints import (
    las_wgs84_bbox, fetch_osm_buildings, iter_building_point_clouds
)
from roof_measurements.pipeline import process_building
import geopandas as gpd
import matplotlib.pyplot as plt

# ── Config ─────────────────────────────────────────────────────────
LAS_OSM = "../data/austin_buildings_sample.laz"  # change to nyc_sandy.laz + epsg=32618
EPSG    = 32614

# 1. Get WGS84 bounding box of the tile
west, south, east, north = las_wgs84_bbox(LAS_OSM, epsg=EPSG)
print(f"Tile bbox: W={west:.5f}  S={south:.5f}  E={east:.5f}  N={north:.5f}")

# 2. Fetch OSM building footprints
footprints = fetch_osm_buildings(west, south, east, north)
print(f"Found {len(footprints)} OSM building footprint(s)")
footprints[['geometry']].head()

In [ ]:
# 3. Visualise footprints over the tile extent
from shapely.geometry import box

tile_box = gpd.GeoDataFrame(
    geometry=[box(west, south, east, north)], crs="EPSG:4326"
)

fig, ax = plt.subplots(figsize=(8, 6))
tile_box.boundary.plot(ax=ax, color='gray', linewidth=1.5, label='LiDAR tile')
if len(footprints) > 0:
    footprints.plot(ax=ax, color='#e63946', alpha=0.5, edgecolor='#c1121f',
                    linewidth=0.8, label='OSM buildings')
ax.set_title(f"OSM footprints in LiDAR tile ({len(footprints)} buildings)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 4. Process each building independently
building_xyz = {}   # keep points for plotting in the next cell
results = []

for bldg_id, xyz, ground_z in iter_building_point_clouds(LAS_OSM, epsg=EPSG, min_points=50):
    building_xyz[bldg_id] = xyz
    try:
        r = process_building(bldg_id, xyz, ground_z,
                             distance_threshold=0.15, min_facet_area_m2=0.5)
        results.append(r)
        mean_pitch = sum(f.pitch_deg for f in r.facets) / r.num_facets
        print(f"  Building {bldg_id[:12]:12s}: "
              f"{r.num_facets} facet(s)  "
              f"pitch={mean_pitch:.1f}°  "
              f"height={r.height_m:.1f}m  "
              f"[{r.segmentation_method}]")
    except Exception as exc:
        print(f"  Building {bldg_id[:12]:12s}: SKIPPED ({exc})")

print(f"\nProcessed {len(results)} / {len(building_xyz)} building(s) successfully.")

In [ ]:
# 5. 3-D facet plots for up to 4 buildings
from roof_measurements.segmentation import segment_planes

n_show = min(len(results), 4)
if n_show == 0:
    print("No results to plot.")
else:
    fig, axes = plt.subplots(
        1, n_show, figsize=(5 * n_show, 4),
        subplot_kw={"projection": "3d"},
    )
    if n_show == 1:
        axes = [axes]

    for ax, r in zip(axes, results[:n_show]):
        xyz  = building_xyz[r.building_id]
        orig = xyz.min(axis=0)          # normalise to local origin
        facet_pts, _ = segment_planes(xyz, distance_threshold=0.15, min_facet_area_m2=0.5)

        for i, fpts in enumerate(facet_pts):
            ax.scatter(
                fpts[:, 0] - orig[0],
                fpts[:, 1] - orig[1],
                fpts[:, 2],
                s=2, c=PALETTE[i % len(PALETTE)],
            )

        ax.set_title(
            f"Bldg {r.building_id[:8]}\n{r.num_facets} facets  h={r.height_m:.1f}m",
            fontsize=9,
        )
        ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)"); ax.set_zlabel("Z (m)")
        ax.view_init(elev=30, azim=-60)

    plt.tight_layout()
    plt.show()

### Export — GeoJSON & CSV

`results_to_geodataframe` joins each `BuildingResult` back to its OSM footprint polygon, giving you a single GeoDataFrame you can write to GeoJSON (for QGIS / Mapbox) or CSV (for spreadsheets / pandas).

Key columns: `height_m` (ridge), `eave_height_m` (lowest eave above ground), `mean_pitch_deg`, `total_roof_area_m2`.

In [ ]:
from roof_measurements.export import results_to_geodataframe, to_geojson, to_csv
from pathlib import Path

gdf = results_to_geodataframe(results, footprints)

# Show the attribute table
display_cols = [
    'osm_id', 'building_type', 'num_facets',
    'height_m', 'eave_height_m', 'mean_pitch_deg', 'total_roof_area_m2',
]
gdf[display_cols].style.format({
    'height_m':        '{:.2f}',
    'eave_height_m':   '{:.2f}',
    'mean_pitch_deg':  '{:.1f}',
    'total_roof_area_m2': '{:.1f}',
})

In [ ]:
# Write to disk
OUT = Path('../data/outputs')

geojson_path = to_geojson(gdf, OUT / 'austin_roofs.geojson')
csv_path     = to_csv(gdf,     OUT / 'austin_roofs.csv', include_wkt=True)

print(f'GeoJSON → {geojson_path}')
print(f'CSV     → {csv_path}')

# Quick sanity-check: reload and verify row count
import geopandas as gpd, pandas as pd
reloaded = gpd.read_file(geojson_path)
print(f'Re-loaded {len(reloaded)} features from GeoJSON ✓')
print(pd.read_csv(csv_path)[['osm_id','height_m','eave_height_m','mean_pitch_deg']].to_string(index=False))